In [0]:
!pip install -U pypdf langchain-text-splitters pandas databricks-langchain

---
## Setup: Install Dependencies
Install required libraries and restart Python to load them.

In [0]:
dbutils.library.restartPython() 

In [0]:
# Import required libraries
import os
import numpy as np
import pandas as pd
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from databricks_langchain import DatabricksEmbeddings, ChatDatabricks

---
## Step 1: Extract Text from PDFs
Load PDF files from the directory and extract text page-by-page.

# Retrieval-Augmented Generation (RAG) on Databricks

A complete end-to-end RAG implementation for building AI applications that answer questions based on your documents.

## What is RAG?
RAG combines document retrieval with generative AI to provide accurate, context-grounded answers. Instead of relying solely on an LLM's training data, RAG retrieves relevant information from your documents first, then uses that context to generate responses.

## The Pipeline
**Phase 1: Preparation** - Chunking → Embedding  
**Phase 2: Execution** - Retrieval → Generation

In [0]:
# Load all PDFs from directory
base_path = os.path.join(os.getcwd(), 'RAG/PDFs')
pages = []

for file_name in os.listdir(base_path):
    full_path = os.path.join(base_path, file_name)
    reader = PdfReader(full_path)
    
    # Extract text from each page
    for page_num, page in enumerate(reader.pages, start=1):
        pages.append({
            "text": page.extract_text(),
            "page_num": page_num
        })

print(f"✓ Loaded {len(pages)} pages from PDF documents")



---
## Step 2: Chunking - Break Documents into Paragraphs

**Why?** Embedding models and LLMs have token limits. Smaller chunks enable precise retrieval.

**Configuration:**
* Chunk size: 1000 characters
* Overlap: 100 characters (preserves context)
* Separators: Paragraph → Line → Word → Comma

In [0]:
# Initialize text splitter with chunking parameters
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ",", ""]
)

# Apply chunking to all pages
chunks = []
for i, page in enumerate(pages):
    page_chunks = splitter.split_text(page["text"])
    
    for j, chunk in enumerate(page_chunks):
        chunks.append({
            "chunk": chunk,
            "id": f'chunk_id__{page["page_num"]}_{j}'
        })

print(f"✓ Created {len(chunks)} chunks from {len(pages)} pages")

---
## Step 3: Embedding - Convert Text to Vectors

**Why?** Embeddings convert text into numeric vectors that capture semantic meaning. Similar texts have similar vectors.

**Configuration:**
* Model: `databricks-bge-large-en` (1024 dimensions)
* Process: Text chunks → Embeddings → Normalized vectors
* Normalization: Makes cosine similarity = dot product (faster)

In [0]:
# Store chunks in DataFrame for easy manipulation
data = pd.DataFrame(chunks)

# Initialize embedding model
embedding_model = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

# Convert all chunks to embeddings (this may take a minute...)
texts = data["chunk"].tolist()
embeddings = np.array(embedding_model.embed_documents(texts))

print(f"✓ Generated embeddings: {embeddings.shape}")
print(f"  - {embeddings.shape[0]} chunks")
print(f"  - {embeddings.shape[1]} dimensions per vector")

In [0]:
# Normalize embeddings for cosine similarity
def normalize(vector):
    norms = np.linalg.norm(vector, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12  # Avoid division by zero
    return vector / norms

chunk_vectors = normalize(embeddings)
print(f"✓ Normalized {len(chunk_vectors)} vectors for similarity search")

---
## Step 4: Retrieval - Find Relevant Chunks

**How it works:**
1. Convert user query to embedding vector
2. Compute cosine similarity with all chunk vectors
3. Rank chunks by similarity score (0=unrelated, 1=identical)
4. Return top-k most relevant chunks

**Math:** Cosine similarity = dot product of normalized vectors

In [0]:
def retrieve(query, k=3):
    """Find the k most relevant chunks for a given query."""
    # 1. Embed query
    query_vector = np.array(embedding_model.embed_query(query)).reshape((1, -1))
    
    # 2. Normalize query vector
    normalized_query = normalize(query_vector)
    
    # 3. Compute similarity scores with all chunks
    scores = (chunk_vectors @ normalized_query.T).flatten()
    
    # 4. Get top-k indices (highest scores first)
    top_indices = np.argsort(scores)[::-1][:k]
    
    # 5. Return matching chunks with metadata
    return [data.iloc[i].to_dict() for i in top_indices]

---
## Step 5: Generation - Create Grounded Answers

**How it works:**
1. Retrieve top-k relevant chunks
2. Build context string from retrieved chunks
3. Create prompt: "Answer using ONLY the provided context"
4. LLM generates grounded response

**Configuration:**
* Model: Llama 3.3 70B Instruct
* Temperature: 0.1 (deterministic, factual)
* Max tokens: 500 (concise answers)

**Why RAG?** Prevents hallucinations by grounding answers in your documents.

In [0]:
# Initialize LLM for answer generation
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=500
)

In [0]:
def create_prompt(retrieved_chunks, question):
    """Build prompt with retrieved context."""
    # Combine all retrieved chunks into context
    context = "\n\n".join([
        f"SOURCE {i+1}:\n{chunk['chunk']}"
        for i, chunk in enumerate(retrieved_chunks)
    ])
    
    # Create instruction prompt
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the information provided in the sources below. If the answer cannot be found in the sources, say so. keep the answer short and precise, two sentences at most.

QUESTION: {question}

SOURCES:
{context}

ANSWER:"""
    
    return prompt

In [0]:
def RAG(query):
    retrieved_sources = retrieve(query, k = 4)
    prompt = create_prompt(retrieved_sources, query)
    response = llm.invoke(prompt)
    answer = response.content

    return {
        "question" : query,
        "retrieved_sources" : retrieved_sources,
        "answer" : answer
    }

---
## Test the RAG System

Now you can ask questions and get answers grounded in your PDF documents.

In [0]:
# Example 1: Query about Data Engineering
RAG("What is data engineering?")

In [0]:
# Example 2: Query about Linux commands
RAG("What is the Linux command to list files?")

In [0]:
RAG("what are the role of engineer?")